# 🌉 Bridge Crack Detection — YOLOv8-Seg Training

This notebook trains a **YOLOv8n-seg** (nano segmentation) model on the **crack-seg** dataset using a **free T4 GPU**.

**Instructions:**
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Click **Runtime → Run all**
3. Wait ~30-60 minutes for training to complete
4. Download `crack_seg_best.pt` from the last cell
5. Place it in your project: `bridge_crack_detection/models/crack_seg_best.pt`

## 1. Setup & Install

In [ ]:
# Verify GPU is available
!nvidia-smi

# Install ultralytics
!pip install -q ultralytics

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Train the Model

This will:
- Auto-download the crack-seg dataset (~92 MB, 4029 images)
- Train YOLOv8n-seg for 100 epochs with early stopping (patience=30)
- Should take **30-60 minutes** on a T4 GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained nano segmentation model
model = YOLO("yolov8n-seg.pt")

# Train on crack-seg dataset
results = model.train(
    data="crack-seg.yaml",
    epochs=100,
    patience=30,
    batch=16,
    imgsz=640,
    device=0,
    cache=True,
    project="runs/segment",
    name="crack_seg",
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True,
)

## 3. View Training Results

In [ ]:
from IPython.display import Image, display
import os

# Show training curves
results_img = "runs/segment/crack_seg/results.png"
if os.path.exists(results_img):
    display(Image(filename=results_img, width=900))
else:
    print("Results plot not found.")

In [ ]:
# Show confusion matrix
cm_img = "runs/segment/crack_seg/confusion_matrix_normalized.png"
if os.path.exists(cm_img):
    display(Image(filename=cm_img, width=600))

In [ ]:
# Show sample validation predictions
val_img = "runs/segment/crack_seg/val_batch0_pred.png"
if os.path.exists(val_img):
    display(Image(filename=val_img, width=900))

## 4. Validate on Test Set

In [ ]:
best_model = YOLO("runs/segment/crack_seg/weights/best.pt")
metrics = best_model.val(data="crack-seg.yaml", split="test", device=0)

print(f"\n{'='*50}")
print(f"  FINAL TEST METRICS")
print(f"{'='*50}")
print(f"  Box  mAP50    : {metrics.box.map50:.4f}")
print(f"  Box  mAP50-95 : {metrics.box.map:.4f}")
print(f"  Mask mAP50    : {metrics.seg.map50:.4f}")
print(f"  Mask mAP50-95 : {metrics.seg.map:.4f}")
print(f"{'='*50}")

## 5. Download Trained Weights

Run the cell below, then place the downloaded file at:
```
bridge_crack_detection/models/crack_seg_best.pt
```
Then restart your backend and the API will auto-load the real model!

In [ ]:
import shutil
from google.colab import files

src = "runs/segment/crack_seg/weights/best.pt"
dst = "crack_seg_best.pt"
shutil.copy2(src, dst)

print(f"Weights ready: {dst}")
print(f"File size: {os.path.getsize(dst) / 1e6:.1f} MB")
print(f"Downloading to your computer...")

files.download(dst)